In [ ]:
!nvidia-smi


Sat Jul 25 02:09:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.2 MB/s eta 0:00:00


In [10]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.8/112.6 GB disk)


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

base = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8"

for split in ["train", "valid", "test"]:
    path = os.path.join(base, split, "images")
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"{split:6s}: {n} ảnh")
    else:
        print(f"{split:6s}: (không tìm thấy thư mục images)")

train : 13300 ảnh
valid : 3725 ảnh
test  : 1900 ảnh


In [15]:
import shutil, os

# Nguồn: dataset trên Google Drive của bạn
src = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8"
# Đích: ổ local tạm của máy chủ Colab (nhanh)
dst = "/content/vehicles_local"

print("⏳ Đang copy dataset vào ổ local của Colab... (chờ vài phút)")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("✅ Copy xong! Dataset giờ ở:", dst)

for split in ["train", "valid", "test"]:
    p = os.path.join(dst, split, "images")
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"{split:6s}: {n} ảnh")

⏳ Đang copy dataset vào ổ local của Colab... (chờ vài phút)
✅ Copy xong! Dataset giờ ở: /content/vehicles_local
train : 13300 ảnh
valid : 3798 ảnh
test  : 1900 ảnh


In [13]:
yaml_content = """train: /content/vehicles_local/train/images
val: /content/vehicles_local/valid/images
test: /content/vehicles_local/test/images

nc: 4
names: ['bus', 'car', 'motorcycle', 'truck']
"""
with open("/content/data_baseline.yaml", "w") as f:
    f.write(yaml_content)
print("✅ Đã cập nhật data_baseline.yaml trỏ vào ổ local")
!cat /content/data_baseline.yaml

✅ Đã tạo lại data_baseline.yaml
train: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8/train/images
val: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8/valid/images
test: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8/test/images

nc: 4
names: ['bus', 'car', 'motorcycle', 'truck']


In [17]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/data_baseline.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    name="baseline_yolov8n",
    patience=10,
)

Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_baseline.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolov8n-4, nbs=64, nms=Fa

In [ ]:
import shutil, os
src_10 = "/content/runs/detect/baseline_yolov8n"
dst_10 = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/baseline_10epochs"
if os.path.exists(src_10):
    shutil.copytree(src_10, dst_10, dirs_exist_ok=True)
    print("✅ Đã lưu kết quả 10 epoch về Drive:", dst_10)
else:
    print("❌ Không tìm thấy nguồn:", src_10)

In [20]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/data_baseline.yaml",
    epochs=20,                    # ← sửa 10 thành 20
    imgsz=640,
    batch=16,
    name="baseline_20epochs",     # ← đổi tên để dễ phân biệt (khuyên dùng)
    patience=10,
    seed=42,
)

Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_baseline.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_20epochs, nbs=64, nms=Fal

In [21]:
import shutil, os

src_20 = "/content/runs/detect/baseline_20epochs"
dst_20 = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/baseline_20epochs"

if os.path.exists(src_20):
    shutil.copytree(src_20, dst_20, dirs_exist_ok=True)
    print("✅ Đã lưu kết quả 20 epoch về Drive!")
    print("📁 Vị trí:", dst_20)
    for f in ["weights/best.pt", "results.png", "results.csv", "confusion_matrix.png"]:
        path = os.path.join(dst_20, f)
        print(f"{'✅' if os.path.exists(path) else '❌'} {f}")
else:
    print("❌ Không tìm thấy nguồn. Kiểm tra:")
    !ls /content/runs/detect/

✅ Đã lưu kết quả 20 epoch về Drive!
📁 Vị trí: /content/drive/MyDrive/Do an AI giao thong/Ketqua_Baseline/baseline_20epochs
✅ weights/best.pt
✅ results.png
✅ results.csv
✅ confusion_matrix.png


In [25]:
# Đánh giá mô hình trên tập TEST (dữ liệu mô hình chưa từng thấy)
metrics_test = model.val(
    data="/content/data_baseline.yaml",
    split="test",              # ← điểm mấu chốt: đánh giá trên tập test
    name="eval_test_20epochs", # tên thư mục lưu kết quả
)

print("\n========== KẾT QUẢ TRÊN TẬP TEST ==========")
print(f"mAP@0.5      : {metrics_test.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test.box.map:.4f}")
print(f"Precision    : {metrics_test.box.mp:.4f}")
print(f"Recall       : {metrics_test.box.mr:.4f}")

Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.3 ms, read: 20.2±8.6 MB/s, size: 47.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8/test/labels... 1900 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1900/1900 63.4it/s 30.0s
val: New cache created: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 4.5it/s 26.3s
                   all       1900       7036      0.719      0.579      0.659      0.458
                   bus        412        642    

In [26]:
import shutil, os

src = "/content/runs/detect/eval_test_20epochs"
dst = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/eval_test_20epochs"

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("✅ Đã lưu kết quả đánh giá test về Drive:", dst)
else:
    print("❌ Không tìm thấy nguồn:", src)

✅ Đã lưu kết quả đánh giá test về Drive: /content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/eval_test_20epochs
